In [1]:
%%capture
!sudo apt-get update
!sudo apt-get install -y xvfb ffmpeg freeglut3-dev
!pip install 'imageio==2.4.0'
!apt-get install python-opengl -y
!apt install xvfb -y
!pip install pyvirtualdisplay
!pip install piglet
!pip install gymnasium
!apt-get install python-opengl -y
!apt install xvfb -y
!pip install -U --no-cache-dir gdown --pre
!pip install minigrid
!wget -q https://www.dropbox.com/scl/fi/jhkb2y3jw8wgin9e26ooc/MiniGrid-MultiRoom-N6-v0_vid.mp4?rlkey=qtkrmmbk9aiote5z7w4bx6ixi&st=zbr4gk21&dl=1 -O content/MiniGrid-MultiRoom-N6-v0_vid.mp4

In [2]:
from pyvirtualdisplay import Display
from IPython.display import HTML
from IPython import display as ipythondisplay
import pyvirtualdisplay
import IPython
import base64
import gymnasium
import minigrid
from minigrid.wrappers import RGBImgObsWrapper, RGBImgPartialObsWrapper, ImgObsWrapper, FullyObsWrapper, RGBImgPartialObsWrapper
import matplotlib.pyplot as plt
import imageio
import numpy as np
import cv2

In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Set device
print(f"Using device: {device}")  # Check if CUDA is available

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
import collections
from collections import deque
import random
import pickle
from tqdm import tqdm
from google.colab import drive
import os
import re

Using device: cuda


In [4]:
# Constants for the environemnt configuration do no change the values
highlight = False
render_mode = "rgb_array"

In [17]:
def embed_mp4(filename):
  """Embeds an mp4 file in the notebook."""
  video = open(filename,'rb').read()
  b64 = base64.b64encode(video)
  tag = '''
  <video width="640" height="480" controls>
    <source src="data:video/mp4;base64,{0}" type="video/mp4">
  Your browser does not support the video tag.
  </video>'''.format(b64.decode())

  return IPython.display.HTML(tag)
display = pyvirtualdisplay.Display(visible=0, size=(1400, 900)).start()

In [6]:
class DuelingDDQN_NN(nn.Module):
    def __init__(self, action_dim):
        super(DuelingDDQN_NN, self).__init__()

        # Shared convolutional layers for feature extraction
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1),  # 1st Conv layer --> input: 1 channel, output: 32 channels
            nn.ReLU(), # Activation function
            nn.Conv2d(32, 64, kernel_size=3, stride=1), # 2nd Conv layer --> input: 32 channel, output: 64 channels
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=1), # 3rd Conv layer --> input: 64 channel, output: 128 channels
            nn.ReLU(),
            nn.Flatten()
        )

        # Advantage stream - A(s, a)
        self.fc_adv = nn.Sequential(
            nn.Linear(128 * 8 * 8, 256), # Fully connected layer
            nn.ReLU(),
            nn.Linear(256, action_dim) # Output Q-value estimate for each possible action the agent can take
        )

        # Value stream - V(s)
        self.fc_val = nn.Sequential(
            nn.Linear(128 * 8 * 8, 256), # Fully connected layer
            nn.ReLU(),
            nn.Linear(256, 1)  # Output single value for state

        )

    def forward(self, x):
        features = self.conv_layers(x)  # Shared feature extraction

        adv = self.fc_adv(features)  # Advantage stream
        val = self.fc_val(features)  # Value stream

        # Combine value and advantage streams
        q_values = val + (adv - adv.mean(dim=1, keepdim=True)) # Q(s,a) = V(s) + A(s) - mean(A(s))
        return q_values

In [20]:
def preprocess_image(state):
  green_channel_state = state[:, :, 1]
  resized_state = cv2.resize(green_channel_state, (14, 14), interpolation=cv2.INTER_AREA) # Resize image from 56x56 to 14x14
  return torch.tensor(resized_state, dtype=torch.float32).unsqueeze(0).to(torch.device("cuda" if torch.cuda.is_available() else "cpu")) / 255.0  # Shape: [1, 14, 14]

def reverse_action_mapping(action):
  mapping = {0: 0, 1: 1, 2: 2, 3: 5}
  return mapping[action]

def load_model(model, path):
  with open(path, "rb") as f:
    model.load_state_dict(pickle.load(f))
  print(f"Model parameters loaded successfully")

def model_evaluation(env, model, num_episodes = 1, video_name = ''):
  model.eval() # Set the model to evaluation mode (disables gradient updates)
  video_filename = f'{video_name}.mp4'
  total_rewards = []
  total_steps = []
  success = 0 # Count successful episodes

  for episode in range(num_episodes):
    with imageio.get_writer(video_filename, fps=24) as video:
      state, _ = env.reset() # Reset environment
      truncated = False
      episode_reward = 0
      episode_step = 0

      while not truncated:
        preprocess_state = preprocess_image(state).to(torch.device("cuda" if torch.cuda.is_available() else "cpu")).unsqueeze(0) # Preprocess the current state

        with torch.no_grad():
          action = model(preprocess_state).argmax().item() # Select the best action using the trained model
        env_action = reverse_action_mapping(action)

        next_state, reward, done, truncated, _ = env.step(env_action) # Execute the action in the environment

        state = next_state # Update the state

        video.append_data(env.render())
        episode_reward += reward
        episode_step += 1

        if done:
          success += 1
          break

      total_rewards.append(episode_reward)
      total_steps.append(episode_step)

      print(f"Episode {episode + 1} - Reward: {episode_reward}, Steps {episode_step}")
      print("-"*50)

  return total_rewards, total_steps, success


Evalute 2 Small Rooms Agent:

In [21]:
env = gymnasium.make("MiniGrid-MultiRoom-N2-S4-v0", render_mode=render_mode, highlight=highlight)
env = RGBImgPartialObsWrapper(env)
env = ImgObsWrapper(env)

model = DuelingDDQN_NN(4).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model_path = "/content/train_2SmallRooms_60000.pkl"
load_model(model, model_path)

num_episodes = 10
video_name = 'Evaluation_MultiRoomEnv_6Rooms'
rewards, steps, success = model_evaluation(env, model, num_episodes, video_name)

print("\nEvaluation Summary:")
print("=" * 50)
print(f"Success Rate: {(success / num_episodes * 100):.1f}%")
print(f"Mean Steps Length: {np.mean(steps):.1f}")
print(f"Mean Total Reward: {np.mean(rewards):.2f}")
print("=" * 50)
print("")

embed_mp4('Evaluation_MultiRoomEnv_6Rooms.mp4')

Model parameters loaded successfully
Episode 1 - Reward: 0.82, Steps 8
--------------------------------------------------
Episode 2 - Reward: 0.865, Steps 6
--------------------------------------------------
Episode 3 - Reward: 0.8875, Steps 5
--------------------------------------------------
Episode 4 - Reward: 0.7975, Steps 9
--------------------------------------------------
Episode 5 - Reward: 0.7975, Steps 9
--------------------------------------------------
Episode 6 - Reward: 0.865, Steps 6
--------------------------------------------------
Episode 7 - Reward: 0.91, Steps 4
--------------------------------------------------
Episode 8 - Reward: 0.7975, Steps 9
--------------------------------------------------
Episode 9 - Reward: 0.82, Steps 8
--------------------------------------------------
Episode 10 - Reward: 0.82, Steps 8
--------------------------------------------------

Evaluation Summary:
Success Rate: 100.0%
Mean Steps Length: 7.2
Mean Total Reward: 0.84



Evalute 6 Small Rooms Agent:

In [22]:
env = gymnasium.make("MiniGrid-MultiRoom-N4-S5-v0", render_mode=render_mode, highlight=highlight)
env = RGBImgPartialObsWrapper(env)
env = ImgObsWrapper(env)

model = DuelingDDQN_NN(4).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model_path = "/content/train_6SmallRooms_80000.pkl"
load_model(model, model_path)

num_episodes = 10
video_name = 'Evaluation_MultiRoomEnv_6Rooms'
rewards, steps, success = model_evaluation(env, model, num_episodes, video_name)

print("\nEvaluation Summary:")
print("=" * 50)
print(f"Success Rate: {(success / num_episodes * 100):.1f}%")
print(f"Mean Steps Length: {np.mean(steps):.1f}")
print(f"Mean Total Reward: {np.mean(rewards):.2f}")
print("=" * 50)
print("")

embed_mp4('Evaluation_MultiRoomEnv_6Rooms.mp4')

Model parameters loaded successfully
Episode 1 - Reward: 0.7975, Steps 27
--------------------------------------------------
Episode 2 - Reward: 0.745, Steps 34
--------------------------------------------------
Episode 3 - Reward: 0.7150000000000001, Steps 38
--------------------------------------------------
Episode 4 - Reward: 0.745, Steps 34
--------------------------------------------------
Episode 5 - Reward: 0.745, Steps 34
--------------------------------------------------
Episode 6 - Reward: 0.7525, Steps 33
--------------------------------------------------
Episode 7 - Reward: 0.7224999999999999, Steps 37
--------------------------------------------------
Episode 8 - Reward: 0.7525, Steps 33
--------------------------------------------------
Episode 9 - Reward: 0.7375, Steps 35
--------------------------------------------------
Episode 10 - Reward: 0.7375, Steps 35
--------------------------------------------------

Evaluation Summary:
Success Rate: 100.0%
Mean Steps Length: 

Evalute 6 Rooms Agent:

In [24]:
env = gymnasium.make("MiniGrid-MultiRoom-N6-v0", render_mode=render_mode, highlight=highlight)
env = RGBImgPartialObsWrapper(env)
env = ImgObsWrapper(env)

model = DuelingDDQN_NN(4).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model_path = "/content/train_6Rooms_90000.pkl"
load_model(model, model_path)

num_episodes = 10
video_name = 'Evaluation_MultiRoomEnv_6Rooms'
rewards, steps, success = model_evaluation(env, model, num_episodes, video_name)

print("\nEvaluation Summary:")
print("=" * 50)
print(f"Success Rate: {(success / num_episodes * 100):.1f}%")
print(f"Mean Steps Length: {np.mean(steps):.1f}")
print(f"Mean Total Reward: {np.mean(rewards):.2f}")
print("=" * 50)
print("")

embed_mp4('Evaluation_MultiRoomEnv_6Rooms.mp4')

Model parameters loaded successfully
Episode 1 - Reward: 0.6174999999999999, Steps 51
--------------------------------------------------
Episode 2 - Reward: 0.7075, Steps 39
--------------------------------------------------
Episode 3 - Reward: 0.73, Steps 36
--------------------------------------------------
Episode 4 - Reward: 0.44499999999999995, Steps 74
--------------------------------------------------
Episode 5 - Reward: 0.61, Steps 52
--------------------------------------------------
Episode 6 - Reward: 0.565, Steps 58
--------------------------------------------------
Episode 7 - Reward: 0.58, Steps 56
--------------------------------------------------
Episode 8 - Reward: 0.6025, Steps 53
--------------------------------------------------
Episode 9 - Reward: 0.5049999999999999, Steps 66
--------------------------------------------------
Episode 10 - Reward: 0.58, Steps 56
--------------------------------------------------

Evaluation Summary:
Success Rate: 100.0%
Mean Steps L